# Project — Building the Perfect Customer Review Model
**Use Case #7: Sentiment Analysis of Customer Reviews (Cognizant / KIET)**

*Step-by-step build. Read top to bottom. Each section explains what we do and why.*

**Aim (one line):** `1 review → auto aspects → feeling for each aspect → one overall feeling (Positive/Negative/Neutral/Mixed)` — no fixed list like `["battery","delivery"]`.

**Example:** `"The product is excellent but delivery was terrible."` → `product→Positive, delivery→Negative, Overall→Mixed`.

---
**Journal Index (how we will go):**

| Entry | Date | What |
|-------|------|------|
| 0 | Day 0 | Setup |
| 1 | Day 1 | Meet the Data (EDA) |
| 2 | Day 2 | Cleaning |
| 3 | Day 3 | Split |
| 4 | Day 4 | Words to Tokens |
| 5 | Day 5 | Labels |
| 6 | Day 6 | Dataset |
| 7 | Day 7 | Model |
| 8 | Day 8 | Training |
| 9 | Day 9 | Scores & Overfit |
| 10 | Day 10 | Use It (Any CSV) |


## 0. Setup
*Before we start, we get our tools. Prepare the environment.*

In [ ]:
# Entry 0: Get tools (run once in Colab)
!pip install -q datasets transformers scikit-learn torch pandas
# datasets → to load DMASTE
# transformers → BERT
# torch → for BERT
# scikit-learn → for scores
# pandas → for tables


**Reflection:** Tools ready. No code is big — each line does one thing.

In [ ]:
# All tools in one place — each line has a comment in clear English
import ast                  # Turn "[('a','b')]" string into a real list
import re                   # Find words with patterns
import pandas as pd         # Work with tables (like Excel)
from datasets import load_dataset  # Load public data
from sklearn.model_selection import train_test_split  # Split cleanly
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report  # Scores
print("All tools ready — small imports, easy to read")


---
## 1. Explore the Data
*First, we inspect the raw data.*


In [ ]:
# Entry 1.1: Load the public data — where did it come from?
def load_raw_data():
    dataset = load_dataset("SilvioLima/raw_data")
    df = dataset["train"].to_pandas()
    return dataset, df

dataset, df = load_raw_data()
print(dataset)  # 13,513 rows
print("Columns:", list(df.columns))  # source, domain, sentence, triples
df.head()


In [ ]:
# Entry 1.2: Where does our data live?
def show_source_counts(df):
    counts = df["source"].value_counts()
    print(counts)  # DMASTE is one of them
    return counts
show_source_counts(df)


In [ ]:
# Entry 1.3: See one real review with your own eyes
def show_one_example(df, idx=0):
    print("Sentence:")
    print(df.iloc[idx]["sentence"])
    print("\nTriples (aspect, opinion, feeling):")
    print(df.iloc[idx]["triples"])

show_one_example(df, 0)


In [ ]:
# Entry 1.4: See five — pattern appears
def show_five_examples(df):
    for i in range(5):
        print(f"--- Review {i+1} ---")
        print(df.iloc[i]["sentence"][:160])
        print("Triples:", df.iloc[i]["triples"])
        print()

show_five_examples(df)


**Note:** Data has `sentence` + `triples` like `[('battery','drains','NEG')]`. We need DMASTE only.

In [ ]:
# Entry 1.5: Keep DMASTE only — 7,524 reviews (our aim's data)
def filter_dmast(df):
    dmast = df[df["source"] == "DMASTE"].copy()
    dmast = dmast[["sentence", "triples"]].copy()
    return dmast

dmast_df = filter_dmast(df)
print("DMASTE reviews:", len(dmast_df))
dmast_df.head()


In [ ]:
# Entry 1.6: String to list (small, easy)
def convert_triples(dmast_df):
    print("Before:", type(dmast_df.iloc[0]["triples"]))  # str
    dmast_df["triples"] = dmast_df["triples"].apply(ast.literal_eval)
    print("After:", type(dmast_df.iloc[0]["triples"]))   # list
    return dmast_df

dmast_df = convert_triples(dmast_df)
print(dmast_df.iloc[0]["triples"][:1])


In [ ]:
# Entry 1.7: How many aspects per review? (small table)
def count_triplets(dmast_df):
    counts = dmast_df["triples"].apply(len)
    print("Average per review:", round(counts.mean(), 2))  # ~3.75
    print("Max in one review:", counts.max())              # 19
    print(counts.describe())
    return counts
count_triplets(dmast_df)


**Note:** 3-4 aspects per review. Max 19. Good to know for later.

In [ ]:
# Entry 1.8: Flatten — one row per aspect (small, easy to read)
def flatten_data(dmast_df):
    rows = []
    for _, row in dmast_df.iterrows():
        sentence = row["sentence"]
        for aspect, opinion, sentiment in row["triples"]:
            rows.append({"text": sentence, "aspect": aspect, "opinion": opinion, "sentiment": sentiment})
    return __import__("pandas").DataFrame(rows)

training_df = flatten_data(dmast_df)
print("Total rows (with hidden):", len(training_df))  # 28,233
print(training_df.head())
print(training_df["sentiment"].value_counts())
print(training_df["sentiment"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# Entry 1.9: How long is the text?
def check_text_length(df):
    df["text_len"] = df["text"].apply(len)
    print(df["text_len"].describe())
    # Most 160-595, max 9951 — BERT will cut at 128

check_text_length(training_df)


In [ ]:
# Entry 1.10: Hidden vs visible (big vs small)
# Hidden = aspect = -1 (not written, e.g., "it was good" — what was good?)
def check_hidden(df):
    visible = df[df["aspect"] != -1]
    hidden = df[df["aspect"] == -1]
    print("Visible (word is there):", len(visible))  # 16,288
    print("Hidden (word not there):", len(hidden))   # 11,945
    return visible, hidden

visible_df, hidden_df = check_hidden(training_df)


**Note:** Big finding: 11,945 hidden. BERT needs a word to point to, so we will drop hidden (small decision, big reason).

---
## 2. Cleaning
*Clean the data.*


In [ ]:
# Entry 2.1: Remove hidden (small, honest)
def remove_hidden(df):
    print("Before:", len(df))
    clean = df[df["aspect"] != -1].copy()
    clean["text"] = clean["text"].astype(str).str.strip()
    clean["aspect"] = clean["aspect"].astype(str).str.strip()
    clean["opinion"] = clean["opinion"].astype(str).str.strip()
    clean = clean[clean["text"] != ""].reset_index(drop=True)
    print("After:", len(clean))  # 16,288
    return clean

training_df = remove_hidden(training_df)
print(training_df["sentiment"].value_counts())


In [ ]:
# Entry 2.2: Summary — what we removed and why
def cleaning_summary(df):
    print("Cleaning done!")
    print("Hidden removed: 11,945 (no word to point to)")
    print("Final rows:", len(df))
    for lab in ["POS","NEG","NEU"]:
        print(lab, ":", len(df[df["sentiment"]==lab]))
    print("POS 79% → accuracy alone will lie, watch F1")

cleaning_summary(training_df)


**Note:** Small clean (one line) keeps model honest. Big clean (like removing punctuation) not needed — BERT needs grammar.

---
## 3. Train / Validation / Test Split
*Split by review to avoid leakage.*


In [ ]:
# Entry 3.1: Split by unique review (small, correct)
def split_by_review(df):
    unique = df["text"].unique()
    print("Unique reviews:", len(unique))  # 7,524
    train_texts, test_texts = train_test_split(unique, test_size=0.2, random_state=42)
    train_texts, val_texts = train_test_split(train_texts, test_size=0.2, random_state=42)
    print("Train:", len(train_texts), "Val:", len(val_texts), "Test:", len(test_texts))
    return train_texts, val_texts, test_texts

train_texts, val_texts, test_texts = split_by_review(training_df)


In [ ]:
# Entry 3.2: Make tables
def make_splits(df, train_texts, val_texts, test_texts):
    train = df[df["text"].isin(train_texts)].reset_index(drop=True)
    val = df[df["text"].isin(val_texts)].reset_index(drop=True)
    test = df[df["text"].isin(test_texts)].reset_index(drop=True)
    return train, val, test

train_df, val_df, test_df = make_splits(training_df, train_texts, val_texts, test_texts)
print("Train rows:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


In [ ]:
# Entry 3.3: Leakage check — must be 0 (small check, big trust)
def check_leakage(train_texts, val_texts, test_texts):
    print("Train-Val overlap:", len(set(train_texts) & set(val_texts)))  # 0
    print("Train-Test overlap:", len(set(train_texts) & set(test_texts)))  # 0
    print("Val-Test overlap:", len(set(val_texts) & set(test_texts)))  # 0
    print("All 0 = no leakage. Good!")

check_leakage(train_texts, val_texts, test_texts)


**Note:** Small split (unique) prevents big lie (same review in train and test).

---
## 4. Tokenization
*BERT uses tokens.*


In [ ]:
# Entry 4.1: Load tokenizer (small)
from transformers import AutoTokenizer
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("Tokenizer: bert-base-uncased")
    return tok
tokenizer = load_tokenizer()

def show_offsets(tokenizer, text):
    enc = tokenizer(text, return_offsets_mapping=True)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    for t, (s,e) in zip(toks[:8], enc["offset_mapping"][:8]):
        print(f"  {t:12} -> '{text[s:e]}' offset=({s},{e})")

show_offsets(tokenizer, train_df.iloc[0]["text"])


In [ ]:
# Entry 4.2: 5 labels — small list, big meaning
def define_labels():
    labels = ["O", "ASPECT", "OPINION_POS", "OPINION_NEG", "OPINION_NEU"]
    label2id = {lab:i for i,lab in enumerate(labels)}
    id2label = {i:lab for lab,i in label2id.items()}
    print(label2id)
    return labels, label2id, id2label
LABELS, label2id, id2label = define_labels()


**Note:** `O` = nothing, `ASPECT` = battery/delivery/chair, `OPINION_*` = excellent/terrible + feeling.

In [ ]:
# Entry 4.3: Small helper — find where a word is
def find_span(text, phrase):
    start = text.lower().find(str(phrase).lower())
    if start == -1: return -1, -1
    return start, start + len(str(phrase))
print(find_span("Battery is great", "Battery"))  # (0,7)


In [ ]:
# Entry 4.4: Small helper — label one token
def label_one_token(start, end, asp_start, asp_end, opi_start, opi_end, sentiment, label2id):
    if start==0 and end==0: return label2id["O"]  # [CLS]/[SEP]
    if opi_start!=-1 and start>=opi_start and end<=opi_end:
        return label2id["OPINION_"+sentiment]
    if asp_start!=-1 and start>=asp_start and end<=asp_end:
        return label2id["ASPECT"]
    return label2id["O"]
print("Helper ready — one token, one decision")


In [ ]:
# Entry 4.5: Main — label all tokens in one review (still small, calls helpers)
def make_token_labels(text, aspect, opinion, sentiment, tokenizer, label2id):
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=128)
    asp_start, asp_end = find_span(text, aspect)
    opi_start, opi_end = find_span(text, opinion)
    labels = []
    for s,e in enc["offset_mapping"]:
        labels.append(label_one_token(s,e, asp_start,asp_end, opi_start,opi_end, sentiment, label2id))
    return enc["input_ids"], enc["attention_mask"], labels
print("Main ready — calls two small helpers")


In [ ]:
# Entry 4.6: See it on one row (big picture from small demo)
def demo_labels(df, idx, tokenizer, label2id, id2label):
    row = df.iloc[idx]
    ids, mask, labs = make_token_labels(row["text"], row["aspect"], row["opinion"], row["sentiment"], tokenizer, label2id)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print("Text:", row["text"][:110])
    print("Aspect:", row["aspect"], "| Opinion:", row["opinion"], "| Feeling:", row["sentiment"])
    print("Tokens with labels (only non-O):")
    for t,l in zip(toks, labs):
        if l != label2id["O"]:
            print(f"  {t:15} -> {id2label[l]}")
demo_labels(train_df, 2, tokenizer, label2id, id2label)


**Note:** Five labels handle aspect and sentiment together.

---
## 5. Dataset
*Prepare PyTorch datasets.*


In [ ]:
# Entry 5.1: One row to model input (small)
import torch
from torch.utils.data import Dataset
def convert_row(row, tokenizer, label2id):
    text = row["text"]
    ids, mask, labs = make_token_labels(text, row["aspect"], row["opinion"], row["sentiment"], tokenizer, label2id)
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=128)
    labs_padded = labs + [label2id["O"]]*(128-len(labs))
    labs_padded = labs_padded[:128]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labs_padded}
print(convert_row(train_df.iloc[0], tokenizer, label2id)["input_ids"][:6])


In [ ]:
# Entry 5.2: All rows (small loop, not big comprehension)
def convert_all(df, tokenizer, label2id):
    out = []
    for _, row in df.iterrows():
        out.append(convert_row(row, tokenizer, label2id))
    return out
train_tokens = convert_all(train_df, tokenizer, label2id)
val_tokens = convert_all(val_df, tokenizer, label2id)
test_tokens = convert_all(test_df, tokenizer, label2id)
print("Train:", len(train_tokens), "Val:", len(val_tokens), "Test:", len(test_tokens))


In [ ]:
# Entry 5.3: PyTorch Dataset (small class)
class ReviewDataset(Dataset):
    def __init__(self, data): self.data=data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {k: torch.tensor(v) for k,v in item.items()}
def build_datasets(train_tokens, val_tokens, test_tokens):
    return ReviewDataset(train_tokens), ReviewDataset(val_tokens), ReviewDataset(test_tokens)
train_dataset, val_dataset, test_dataset = build_datasets(train_tokens, val_tokens, test_tokens)
print("Datasets:", len(train_dataset), len(val_dataset), len(test_dataset))


**Note:** Dataset ready.

---
## 6. Model
*Token classification model.*


In [ ]:
# Entry 6.1: Device — GPU or CPU (small check, big safety)
import torch
from transformers import AutoModelForTokenClassification
def get_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device, "| PyTorch:", torch.__version__)
    return device
device = get_device()


In [ ]:
# Entry 6.2: Load model (small)
def load_model(num_labels, device):
    model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    model.to(device)
    print("Model on", device)
    return model
model = load_model(len(LABELS), device)


**Note:** `bert-base-uncased` (110M). Small code `from_pretrained` loads big knowledge.

---
## 7. Training
*Train the model.*


In [ ]:
# Entry 7.1: Settings (small, standard)
from transformers import TrainingArguments, Trainer
def get_training_args():
    return TrainingArguments(
        output_dir="./bert_aste",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=2e-5,
        load_best_model_at_end=True
    )
args = get_training_args()
print(args)


In [ ]:
# Entry 7.2: How to measure? (small, honest)
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1).flatten()
    labels = pred.label_ids.flatten()
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
print("We track weighted F1 — accuracy alone is misleading due to class imbalance (POS 79%)")


In [ ]:
# Entry 7.3: Trainer (small)
def build_trainer(model, args, train_dataset, val_dataset):
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
    print("Trainer ready. Run trainer.train() on T4 (15-20 min)")
    return trainer
trainer = build_trainer(model, args, train_dataset, val_dataset)
# trainer.train()  # <-- Remove # to start


In [ ]:
# Entry 7.4: Save (small)
def save_model(trainer, tokenizer):
    trainer.save_model("./bert_aste_final")
    tokenizer.save_pretrained("./bert_aste_final")
    print("Saved to ./bert_aste_final — check size with !du -sh bert_aste_final/ (expect ~400 MB for BERT, ~250 MB for DistilBERT)")
# save_model(trainer, tokenizer)  # After training
print("Save ready")


**Note:** Training configuration. `eval_strategy` fixed for transformers 4.57.6.

---
## 8. Evaluation
*Evaluate and check for overfitting.*


In [ ]:
# Entry 8.1: Val scores (small)
def show_val_scores(trainer):
    result = trainer.evaluate()
    print(result)
    return result
# show_val_scores(trainer)  # After training
print("Example: {'eval_f1': 0.875, 'eval_accuracy': 0.89}")


In [ ]:
# Entry 8.2: Per label (small, detailed)
def show_detailed_report(trainer, test_dataset):
    preds = trainer.predict(test_dataset)
    y_pred = preds.predictions.argmax(-1).flatten()
    y_true = preds.label_ids.flatten()
    print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))
# show_detailed_report(trainer, test_dataset)
print("Example:")
print("  O            0.96 0.98 0.97")
print("  ASPECT       0.78 0.75 0.76  <- your aim")
print("  OPINION_POS  0.82 0.80 0.81")


**Which score to watch? (Big decision)**
- **Not accuracy alone** — 79% POS fools it.
- **F1 weighted** — main.
- **ASPECT F1** — your aim.
- **OPINION_NEG recall** — don't miss bad.


In [ ]:
# Entry 8.3: Overfit check (small, crucial)
def check_overfit(trainer):
    train_f1 = trainer.evaluate(eval_dataset=train_dataset)["eval_f1"]
    val_f1 = trainer.evaluate(eval_dataset=val_dataset)["eval_f1"]
    print(f"Train F1: {train_f1:.3f}, Val F1: {val_f1:.3f}, Gap: {train_f1-val_f1:.3f}")
    if train_f1 - val_f1 > 0.10:
        print("Overfit — memorizing. Fix: early stop, dropout")
    elif val_f1 < 0.70 and train_f1 < 0.70:
        print("Underfit — too simple. Fix: train longer")
    else:
        print("Good balance")
    return train_f1, val_f1
# check_overfit(trainer)
print("Gap > 0.10 indicates overfitting")


In [ ]:
# Entry 8.4: If accuracy is 0.99 (small sanity, big warning)
def sanity_check():
    print("If 0.99:")
    print("1. Did you split by review? (We did, leakage 0)")
    print("2. Check per-label F1, not just accuracy")
    print("3. Run check_overfit()")
sanity_check()


**Note:** Checks reveal the truth. Gap tells overfit.

---
## 9. Inference
*Run on any review.*


In [ ]:
# Entry 9.1: Clean token
def clean_token(tok): return tok.replace("##", "")
# Entry 9.2: Tokens to aspects
def decode_predictions(toks, preds, id2label):
    aspects, opinions = [], []
    cur_text, cur_label = "", None
    for tok, lab in zip(toks, preds):
        if tok in ["[CLS]","[SEP]","[PAD]"]: continue
        lab_str = id2label[lab]
        is_sub = tok.startswith("##")
        tok_c = clean_token(tok)
        if lab_str == "ASPECT":
            if cur_label != "ASPECT":
                if cur_text:
                    if cur_label=="ASPECT": aspects.append(cur_text)
                    elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text = tok_c
            else: cur_text += tok_c if is_sub else " " + tok_c
            cur_label = "ASPECT"
        elif lab_str.startswith("OPINION"):
            if cur_label != lab_str:
                if cur_text:
                    if cur_label=="ASPECT": aspects.append(cur_text)
                    elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text = tok_c
            else: cur_text += tok_c if is_sub else " " + tok_c
            cur_label = lab_str
        else:
            if cur_text:
                if cur_label=="ASPECT": aspects.append(cur_text)
                elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
                cur_text=""; cur_label=None
    if cur_text:
        if cur_label=="ASPECT": aspects.append(cur_text)
        elif cur_label and cur_label.startswith("OPINION"): opinions.append((cur_text, cur_label))
    return aspects, opinions
print("Decode ready")


In [ ]:
# Entry 9.3: Build triplets + overall (small)
def build_triplets(aspects, opinions):
    triplets=[]
    for asp in aspects:
        sent = opinions[0][1].split("_")[1] if opinions else "NEU"
        m={"POS":"Positive","NEG":"Negative","NEU":"Neutral"}
        triplets.append({"aspect": asp, "sentiment": m[sent]})
    return triplets
def get_overall(triplets):
    pos=sum(1 for t in triplets if t["sentiment"]=="Positive")
    neg=sum(1 for t in triplets if t["sentiment"]=="Negative")
    if pos>0 and neg>0: return "Mixed"
    if pos>0: return "Positive"
    if neg>0: return "Negative"
    return "Neutral"
print("Triplet helpers ready")


In [ ]:
# Entry 9.4: Main predict (small, calls helpers above)
def predict_review(text):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    enc = {k: v.to(device) for k,v in enc.items()}
    with __import__("torch").no_grad():
        out = model(**enc)
        preds = out.logits.argmax(-1)[0].cpu().tolist()
        ids = enc["input_ids"][0].cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    aspects, opinions = decode_predictions(toks, preds, id2label)
    triplets = build_triplets(aspects, opinions)
    return {"review": text, "overall": get_overall(triplets), "aspects": triplets, "raw_aspects": aspects, "raw_opinions": opinions}
print("predict_review() ready")


In [ ]:
# Entry 9.5: Try (after training, remove #)
# print(predict_review("The product is excellent but delivery was terrible."))
# print(predict_review("Battery drains fast but camera is amazing."))
# print(predict_review("The chair armrest is wobbly but fabric is comfortable."))
print("Demo ready")


**Note:** Helpers combine into the main prediction.

---
## 10. Inference on Any CSV
*Supports any CSV column name.*


In [ ]:
# Entry 10.1: Find text column (small)
import csv, io
from collections import Counter
def find_text_column(fieldnames):
    candidates = ["review_text","review text","review","comment","feedback","text","sentence"]
    norm = {h.lower().strip(): h for h in fieldnames}
    for c in candidates:
        if c in norm: return norm[c]
    for h in fieldnames:
        for c in candidates:
            if c in h.lower(): return h
    return fieldnames[0]
print("Column finder ready")


In [ ]:
# Entry 10.2: Read any CSV (small)
def analyze_csv_bytes(content: bytes):
    text = content.decode("utf-8")
    reader = __import__("csv").DictReader(__import__("io").StringIO(text))
    col = find_text_column(reader.fieldnames)
    print("Found column:", col)
    results=[]
    for row in reader:
        txt=(row.get(col) or "").strip()
        if not txt: continue
        results.append(predict_review(txt))
    from collections import Counter
    overall = Counter(r["overall"] for r in results)
    print("Overall:", dict(overall))
    all_a=[a["aspect"] for r in results for a in r["aspects"]]
    print("Top concerns:", Counter(all_a).most_common(5))
    return results
print("CSV analyzer ready")
print("Use: analyze_csv_bytes(open('chair.csv','rb').read())")


---
## Summary

* **Concise code, clear result:** 60 cells, each 5-15 lines, clear English, no `!pip` beyond setup.
* **Key decisions:** Drop hidden `-1`, split by review, watch F1 not accuracy.
* **Modular helpers:** `find_span + label_one_token + make_token_labels` → `predict_review` for any product.

**Next:** Colab → T4 → Run all → `trainer.train()` → `!du -sh bert_aste_final/` → size ~400M (`distilbert` ~250M) → copy to `Cognizant/bert_aste_final/` → deploy.

*End of Project — 10 Days, One Flow, One Model.*
